In [1]:
from pathlib import Path

import polars as pl
from tscbench.utils import S3FileCache

In [2]:
#import boto3
#s3 = boto3.client("s3")
#s3.create_bucket(Bucket="tsc-bench")
#s3.put_object(Bucket="tsc-bench", Key="performance-benchmarking/.keep", Body=b"")


In [3]:
S3_URI = "s3://tsc-bench/performance-benchmarking"
S3_CACHE_DIR = Path("artifacts/s3-cache")

results = S3FileCache(S3_URI).read_all_parquet_cached(cache_dir=S3_CACHE_DIR)
results

100%|██████████| 24/24 [00:00<00:00, 34.97it/s]


dataset,fold,model,random_state,status,python_version,n_jobs,tscbench_version,tscglue_version,aeon_version,sklearn_version,numpy_version,polars_version,n_train,n_test,n_classes,fit_seconds,fit_seconds_per_sample,predict_seconds,inference_seconds,predict_seconds_per_sample,inference_seconds_per_sample,total_seconds,test_accuracy
str,i64,str,i64,str,str,i64,str,str,str,str,str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""ArrowHead""",0,"""catch22""",0,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",36,175,3,0.670711,0.018631,0.260155,0.260155,0.001487,0.001487,0.930866,0.702857
"""ArrowHead""",1,"""catch22""",1,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",36,175,3,0.49617,0.013783,0.255419,0.255419,0.00146,0.00146,0.751589,0.72
"""ArrowHead""",2,"""catch22""",2,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",36,175,3,0.492902,0.013692,0.252358,0.252358,0.001442,0.001442,0.745261,0.725714
"""ArrowHead""",3,"""catch22""",3,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",36,175,3,0.489772,0.013605,0.259882,0.259882,0.001485,0.001485,0.749653,0.748571
"""ArrowHead""",0,"""rocket""",0,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",36,175,3,0.618936,0.017193,1.461565,1.461565,0.008352,0.008352,2.080501,0.822857
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GunPoint""",3,"""catch22""",3,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",50,150,2,0.488393,0.009768,0.177967,0.177967,0.001186,0.001186,0.666361,0.933333
"""GunPoint""",0,"""rocket""",0,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",50,150,2,0.34259,0.006852,0.811643,0.811643,0.005411,0.005411,1.154233,1.0
"""GunPoint""",1,"""rocket""",1,"""ok""","""3.13.2""",8,"""0.1.0""","""0.1.2""","""1.4.0""","""1.7.2""","""2.3.5""","""1.40.1""",50,150,2,0.335527,0.006711,0.825323,0.825323,0.005502,0.005502,1.16085,1.0


In [4]:
required = {"dataset", "fold", "model", "status"}
missing = required - set(results.columns)
if missing:
    raise ValueError(f"Missing required result columns: {sorted(missing)}")

ucr_runs = (
    results
    .filter(~pl.col("dataset").str.starts_with("m-"))
    .filter(pl.col("fold").is_between(0, 29))
    .filter(pl.col("status") == "ok")
    .select("model", "dataset", "fold")
    .unique()
)

coverage = (
    ucr_runs
    .group_by("model")
    .agg(
        pl.len().alias("unique_ucr_dataset_fold_runs"),
        pl.col("dataset").n_unique().alias("unique_ucr_datasets"),
        pl.col("fold").n_unique().alias("unique_folds_seen"),
    )
    .with_columns(
        (pl.col("unique_ucr_datasets") * 30).alias("expected_dataset_fold_runs")
    )
    .with_columns(
        (
            pl.col("unique_ucr_dataset_fold_runs")
            / pl.col("expected_dataset_fold_runs")
            * 100
        ).round(2).alias("coverage_pct")
    )
    .sort("model")
)

if coverage.is_empty():
    print("No successful UCR dataset/fold result rows found.")
else:
    for row in coverage.iter_rows(named=True):
        print(
            f"{row['model']}: "
            f"{row['unique_ucr_dataset_fold_runs']}/"
            f"{row['expected_dataset_fold_runs']} unique UCR dataset-fold runs "
            f"({row['unique_ucr_datasets']} datasets x 30 folds, "
            f"{row['coverage_pct']}%)"
        )

coverage

catch22: 12/90 unique UCR dataset-fold runs (3 datasets x 30 folds, 13.33%)
rocket: 12/90 unique UCR dataset-fold runs (3 datasets x 30 folds, 13.33%)


model,unique_ucr_dataset_fold_runs,unique_ucr_datasets,unique_folds_seen,expected_dataset_fold_runs,coverage_pct
str,u32,u32,u32,u32,f64
"""catch22""",12,3,4,90,13.33
"""rocket""",12,3,4,90,13.33


In [5]:
l = [2,4,6,8,10]
print(l)
l[:4]

[2, 4, 6, 8, 10]


[2, 4, 6, 8]

In [ ]:
2**14

16384

: 